# Confirmatory Factor Analysis (CFA) & Structural Equation Models (SEM)

1. Confirmatory Factor Analysis (CFA)
    1. Cronbach's alpha
    2. Item reliability
    3. Composite reliability
    4. Average Variance Extracted (AVE)
    5. Discriminant validity
2. Structural Equation Models (SEM)
3. Model Fit CFA & SEM

In [ ]:
# Imports
%pip install -q pingouin
%pip install -q --no-deps factor-analyzer semopy

import numpy as np
import pandas as pd
import pingouin as pg

from semopy import Model, calc_stats
from factor_analyzer import FactorAnalyzer

In [ ]:
# Create dataset

df = pd.DataFrame({
    "Obs.": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "i1": [4, 3, 4, 4, 3, 3, 3, 4, 2, 2],
    "i2": [4, 1, 5, 4, 5, 1, 3, 5, 2, 2],
    "i3": [5, 3, 5, 4, 4, 2, 3, 5, 3, 3],
    "Factor Score": [13, 7, 14, 12, 12, 6, 9, 14, 8, 7]
})

## 1. Confirmatory Factor Analysis

The latent factor brand loyalty is measured using these three observed items (in practice, go for established items!), all on a 5-point Likert scale. Responses from 10 participants are provided in the table:

In [ ]:
print(df)

### 1.1 Cronbach's alpha

- Cronbach's alpha evaluates the reliability of a set of items derived from survey data, indicating how well they measure the same underlying latent construct, thereby focusing esp. on internal consistency.
- It ensures that variations in responses are due to the construct being measured, not random error or inconsistencies in the items.
- Cronbach's alpha tells you if people who answer “high" on Question 1 also tend to answer “high" on Questions 2 and 3.

$$
\alpha = \frac{k}{k-1}
\left(
1 - \frac{\sum_{i=1}^{k} \sigma_i^2}{\sigma_X^2}
\right)
$$

**Cronbach's Alpha – Interpretation**

- **> 0.90 – Excellent** *(Potential Redundancy)*
  *Caution:* If $\alpha > 0.95$, your items may be redundant (asking essentially the same thing).

- **0.80–0.90 – Good**
  Ideally where you want to be.

- **0.70–0.80 – Acceptable**
  The most common minimum threshold for research.

- **0.60–0.70 – Questionable**
  Often accepted in *exploratory* research or for new scales.

- **< 0.60 – Poor / Unacceptable**
  The items may not be measuring the same construct.

In [ ]:
# Select items
items = df[["i1", "i2", "i3"]]

# Calculate the variance of each item
item_variances = items.var(ddof=1)

# Calculate the sum of the item variances
sum_item_variances = item_variances.sum()

# Calculate the total score from the three items
total_score = items.sum(axis=1)

# Calculate the variance of the total score
total_variance = total_score.var(ddof=1)

# Calculate the variance of the existing factor score
factor_score_variance = df["Factor Score"].var(ddof=1)

# Calculate Cronbach's alpha and its 95% confidence interval
alpha, confidence_interval = pg.cronbach_alpha(data=items)

# Output
print("=== Item Variances ===")
for item, variance in item_variances.items():
    print(f"{item}: {variance:.4f}")

print(f"\nSum of item variances: {sum_item_variances:.4f}")
print(f"Total score variance (sum of items): {total_variance:.4f}")
print(f"Factor score variance: {factor_score_variance:.4f}")

print("\n=== Cronbach's Alpha ===")
print(f"Number of items (k): {items.shape[1]}")
print(f"Cronbach's alpha: {alpha:.4f}")
print(f"95% confidence interval: {confidence_interval}")

### 1.2 Item reliability

- Item reliability evaluates the performance of individual items
- It tells you the percentage of variation in that item that is caused by your latent construct.
- If a question on a survey consistently receives divergent answers unrelated to the construct, its low reliability indicates it's not contributing to consistent results.
$$
IR_i = \lambda_i^2
$$

**Item Reliability – Interpretation**

| Item Reliability ($R^2$) | Implied Factor Loading ($\lambda$) | Interpretation |
|:---|:---|:---|
| **> 0.50** | **> 0.70** | **Good / Ideal.** The item has more signal than noise. Ideally, you want all items here. |
| **0.25–0.50** | **0.50–0.70** | **Acceptable.** Often kept if the overall Composite Reliability is good, but these items are "noisy". Delete if this improves CR or AVE. |
| **< 0.25 / 0.16** | **< 0.50 / 0.40** | **Unacceptable.** The item is mostly measuring error. It should usually be removed. |

In [ ]:
# Define the CFA model with one latent factor and three items
cfa_model = """
Factor =~ i1 + i2 + i3
"""

# Select the observed items
items = df[["i1", "i2", "i3"]]

# Fit the CFA model
model = Model(cfa_model)
model.fit(items)

# Extract standardized parameter estimates
estimates = model.inspect(std_est=True)

# Extract the standardized factor loadings
loadings = estimates.loc[
    (estimates["op"] == "~") &
    (estimates["rval"] == "Factor"),
    ["lval", "Est. Std"]
].copy()

# Rename columns
loadings.columns = ["Item", "Standardized Factor Loading"]

# Calculate item reliability by squaring the standardized factor loadings
loadings["Item Reliability"] = (
    loadings["Standardized Factor Loading"] ** 2
)

# Reset the index
loadings = loadings.reset_index(drop=True)

# Output
print("\nStandardized Factor Loadings and Item Reliability")
print("-----------------------------")
print(
    loadings.to_string(
        index=False,
        formatters={
            "Standardized Factor Loading": "{:.4f}".format,
            "Item Reliability": "{:.4f}".format
        }
    )
)

### 1.3 Composite reliability

- Similar purpose and interpretation as Cronbach’s alpha
- Takes into account factor loadings (as opposed to Cronbach’s alpha, which assumes equal loadings for all items)
$$
CR =
\frac{
\left(\sum_{i=1}^{k} \lambda_i\right)^2
}{
\left(\sum_{i=1}^{k} \lambda_i\right)^2
+
\sum_{i=1}^{k} \theta_i
}
$$
- **Interpretation and cutoffs as with Cronbach’s alpha.**

In [ ]:
# Calculate error variances for standardized items
loadings["Error Variance"] = (
    1 - loadings["Standardized Factor Loading"] ** 2
)

# Calculate the sum of standardized factor loadings and error variances
sum_loadings = loadings["Standardized Factor Loading"].sum()
sum_error_variances = loadings["Error Variance"].sum()

# Calculate composite reliability
cr = (
    sum_loadings ** 2 /
    (sum_loadings ** 2 + sum_error_variances)
)

# Output
print("\nFactor Loadings and Error Variances")
print("-----------------------------")
print(loadings.to_string(
    index=False,
    float_format=lambda x: f"{x:.4f}"
))

print("\nComposite Reliability")
print("-----------------------------")
print(f"Sum of standardized factor loadings: {sum_loadings:.4f}")
print(f"Squared sum of factor loadings: {sum_loadings ** 2:.4f}")
print(f"Sum of error variances: {sum_error_variances:.4f}")
print(f"Composite Reliability (CR): {cr:.4f}")

### 1.4 Average Variance Extracted (AVE)

- AVE assesses how much of the variance in the observed variables is explained by the underlying construct.
- A higher AVE indicates that the construct explains a substantial portion of the item variances, supporting (converging) construct validity.
- AVE ensures that the items collectively reflect the essence of the construct they are meant to measure, validating the comprehensiveness of the construct.
$$
AVE = \frac{\sum_{i=1}^{k} \lambda_i^2}{k}
$$

$$
= \text{Average Item Reliability}
$$
- **An AVE ≥ 0.50 indicates that more than 50% of the variance in the observed variables is due to the construct, suggesting good validity.**

In [ ]:
# Calculate AVE
ave = loadings["Item Reliability"].mean()

# Output
print("\nAverage Variance Extracted")
print("-----------------------------")
print(f"Number of items: {len(loadings)}")
print(f"Sum of squared factor loadings: {loadings['Item Reliability'].sum():.4f}")
print(f"Average Variance Extracted (AVE): {ave:.4f}")


### 1.5 Discriminant validiy

- Discriminant validity ensures that the construct being measured is distinct and not overlapping with other constructs.
- This supports construct validity by verifying that the instrument accurately captures the intended concept and not something else.
- Key method to measure discriminant validity: Fornell-Larcker criterion
 = checks that the square root of AVE for a construct is greater than its correlation with other constructs, ensuring distinctiveness
- Also relevant, but not covered here: Heterotrait-Monotrait Ratio of Correlations (HTMT)
- Fast but imprecise: Correlations between constructs should not be too high (< 0,85)
- Example: If a model includes constructs like "customer satisfaction" and "customer loyalty," discriminant validity ensures they measure different aspects of customer behavior and are not conflated.

In [ ]:
# Creating two latent construct
df2 = pd.DataFrame({
    "Obs.": list(range(1, 11)),

    # Brand Loyalty
    "i1": [4, 2, 5, 3, 4, 1, 3, 2, 4, 2],
    "i2": [3, 2, 5, 3, 5, 1, 3, 3, 3, 3],
    "i3": [4, 2, 5, 3, 4, 1, 3, 4, 4, 3],

    # Brand Satisfaction
    "i4": [2, 4, 4, 3, 2, 2, 3, 3, 4, 3],
    "i5": [3, 4, 3, 3, 3, 2, 4, 4, 5, 4],
    "i6": [3, 4, 4, 4, 3, 2, 4, 3, 5, 5],
})

# Define the items belonging to each construct
loyalty_items = ["i1", "i2", "i3"]
satisfaction_items = ["i4", "i5", "i6"]
items = loyalty_items + satisfaction_items

In [ ]:
# Fit a two-factor model without rotation
fa = FactorAnalyzer(n_factors=2, method="ml", rotation=None)
fa.fit(df2[items])

# Extract the factor loadings and assign descriptive labels
loadings = pd.DataFrame(
    fa.loadings_,
    index=items,
    columns=["Factor 1", "Factor 2"]
)

# Assign each factor to the construct based on its item loadings
print("\nFactor Loadings:")
print(loadings.round(3))

# Calculate the Average Variance Extracted (AVE) for each construct using the squared loadings of the assigned factor
ave = pd.Series({
    "Brand Loyalty": np.mean(loadings.loc[loyalty_items, "Factor 1"] ** 2),
    "Brand Satisfaction": np.mean(loadings.loc[satisfaction_items, "Factor 2"] ** 2)
})

# Calculate the square root of AVE for the Fornell-Larcker criterion
sqrt_ave = np.sqrt(ave)

# Calculate construct scores as the mean of their respective items
construct_scores = pd.DataFrame({
    "Brand Loyalty": df2[loyalty_items].mean(axis=1),
    "Brand Satisfaction": df2[satisfaction_items].mean(axis=1)
})

# Calculate the correlation between the construct scores
R = construct_scores.corr()

# Create a copy of the correlation matrix for the Fornell-Larcker assessment
fornell_larcker = R.copy(deep=True)

# Replace the diagonal elements with the square root of AVE
for i, construct in enumerate(fornell_larcker.columns):
    fornell_larcker.iloc[i, i] = sqrt_ave.loc[construct]

# Output
print("\nFornell-Larcker Criterion:")
print("-----------------------------")
print(fornell_larcker.round(3))

# Check whether the criterion is satisfied for both constructs
correlation = abs(R.loc["Brand Loyalty", "Brand Satisfaction"])

for construct in sqrt_ave.index:
    passed = sqrt_ave[construct] > correlation
    print(
        f"{construct}: sqrt(AVE) = {sqrt_ave[construct]:.3f}, "
        f"|correlation| = {correlation:.3f} -> "
        f"{'Satisfied' if passed else 'Not satisfied'}")

## 2. Structural Equation Models (SEM)
Estimate a Structural Equation Model (SEM) to investigate whether Brand Experience positively influences Brand Satisfaction and Brand Trust, and whether Brand Satisfaction and Brand Trust positively influence Brand Loyalty.

In [ ]:
# Generate data
# Set the random seed to ensure reproducible results
np.random.seed(42)

# Define the number of observations to generate
n = 500

# Generate latent variables and specify the structural relationships, Brand Experience serves as the exogenous latent variable
brand_experience = np.random.normal(0, 1, n)

# Generate Brand Satisfaction as a function of Brand Experience plus random error
brand_satisfaction = (
    0.7 * brand_experience
    + np.random.normal(0, 0.7, n)
)

# Generate Brand Trust as a function of Brand Experience plus random error
brand_trust = (
    0.6 * brand_experience
    + np.random.normal(0, 0.8, n)
)

# Generate Brand Loyalty as a function of Brand Satisfaction and Brand Trust, plus random error
brand_loyalty = (
    0.5 * brand_satisfaction
    + 0.4 * brand_trust
    + np.random.normal(0, 0.7, n)
)

# Generate observed measurement items for a latent variable
def generate_items(latent, prefix, n_items=3):
    items = {}
    for i in range(1, n_items + 1):
        # Add measurement error to simulate imperfect indicators
        values = latent + np.random.normal(0, 0.5, n)

        # Convert continuous values into integer responses on a 1-7 Likert scale, centered around 4
        values = np.clip(np.rint(4 + values), 1, 7).astype(int)

        # Store the generated item using its designated prefix and index
        items[f"{prefix}{i}"] = values
    return items

# Generate observed items for each latent construct
data = {}
data.update(generate_items(brand_loyalty, "i"))
data.update(generate_items(brand_satisfaction, "s"))
data.update(generate_items(brand_trust, "t"))
data.update(generate_items(brand_experience, "e"))

# Combine all generated items into a single DataFrame
df2 = pd.DataFrame(data)

In [ ]:
# Specify the measurement and structural relationships using lavaan-style model syntax
model_desc = """
    # Measurement model: define each latent construct
    # using its three observed indicators
    Brand_Loyalty =~ i1 + i2 + i3
    Brand_Satisfaction =~ s1 + s2 + s3
    Brand_Trust =~ t1 + t2 + t3
    Brand_Experience =~ e1 + e2 + e3

    # Structural model: specify the hypothesized relationships
    # between the latent constructs
    Brand_Satisfaction ~ Brand_Experience
    Brand_Trust ~ Brand_Experience
    Brand_Loyalty ~ Brand_Satisfaction + Brand_Trust
"""

# Initialize the structural equation model from its specification
model = Model(model_desc)

# Fit the model to the generated observed data
model.fit(df2)

# Inspect the estimated model parameters, including factor loadings, regression coefficients, and relevant variance parameters
print("\nParameter Estimates")
print("-----------------------------")
print(model.inspect())

# Calculate model fit statistics and Output
stats = calc_stats(model)
print("\nModel Fit Statistics")
print("-----------------------------")
print(stats.T)

## 3. Model Fit CFA & SEM

**Comparative Fit Index (CFI)**
- CFI compares the fit of the hypothesized model to that of an "independence model" (a baseline model where all variables are uncorrelated).
- It considers the improvement in fit while penalizing model complexity.
- CFI is less affected by sample size compared to some other indices (e.g., Chi-Square).
- ≥0,90: Acceptable fit

**Tucker-Lewis Index (TLI)**
- TLI, also known as the Non-Normed Fit Index (NNFI), is similar to CFI but penalizes model complexity more stringently.
- It compares the hypothesized model to a null model but adjusts for the degrees of freedom.
- TLI is more stringent than CFI, making it a conservative measure of fit.
- It is particularly sensitive to overfitting.
- ≥0,90: Acceptable fit

**Root Mean Square Error of Approximation (RMSEA)**
- RMSEA measures how well the model, with its specified number of factors, approximates the covariance structure in the population.
- It accounts for model complexity by penalizing overfitting.
- RMSEA is sensitive to model misspecification.
- ≤0,08: Acceptable fit

**Standardized Root Mean Square Residual (SRMR)**
- SRMR is the standardized difference between the observed correlations and those predicted by the model.
- It represents the average discrepancy per element of the covariance matrix.
- SRMR provides a direct measure of the residuals, offering intuitive insight into model adequacy.
- ≤0,08: Acceptable fit

In [ ]:
# Extract model fit indices from the fitted model
stats = calc_stats(model)

# Retrieve the Comparative Fit Index (CFI), Tucker-Lewis Index (TLI), and Root Mean Square Error of Approximation (RMSEA)
cfi = stats.loc["Value", "CFI"]
tli = stats.loc["Value", "TLI"]
rmsea = stats.loc["Value", "RMSEA"]

# Calculate the Standardized Root Mean Square Residual (SRMR)
# Identify the observed variables included in the model
observed_vars = list(model.vars["observed"])
# Calculate the observed covariance matrix and convert it to a NumPy array
observed_cov = df2[observed_vars].cov().to_numpy()
# Convert the observed covariance matrix into a correlation matrix
observed_sd = np.sqrt(np.diag(observed_cov))
observed_corr = observed_cov / np.outer(observed_sd, observed_sd)
# Obtain the model-implied covariance matrix
implied_cov = model.calc_sigma()[0]
# Convert the model-implied covariance matrix into a correlation matrix
implied_sd = np.sqrt(np.diag(implied_cov))
implied_corr = implied_cov / np.outer(implied_sd, implied_sd)
# Calculate the residuals between the observed and model-implied correlations
residuals = observed_corr - implied_corr
# Calculate SRMR as the root mean square of the correlation residuals
srmr = np.sqrt(np.mean(residuals**2))

# Compile the model fit indices and their reference thresholds into a DataFrame
fit_results = pd.DataFrame({
    "Fit Index": ["CFI", "TLI", "RMSEA", "SRMR"],
    "Value": [cfi, tli, rmsea, srmr],
    "Criterion": [">= 0.90", ">= 0.90", "<= 0.08", "<= 0.08"]
})

# Define evaluation criteria for each model fit index
criteria = {
    "CFI": lambda x: x >= 0.90,
    "TLI": lambda x: x >= 0.90,
    "RMSEA": lambda x: x <= 0.08,
    "SRMR": lambda x: x <= 0.08
}

# Evaluate each fit index against its predefined threshold and assign a corresponding status
fit_results["Status"] = [
    "Satisfied" if criteria[index](value) else "Not Satisfied"
    for index, value in zip(
        fit_results["Fit Index"], fit_results["Value"]
    )
]

# Output
print("\nModel Fit Indices:")
print("-----------------------------")
print(fit_results.to_string(index=False))